# 03 — Gold Layer
Build summary tables from the cleaned data. These feed directly into the charts.

## Load and prepare

In [1]:
import pandas as pd
import numpy as np

tickets   = pd.read_csv("it_support_tickets.csv", parse_dates=["created_at"])
customers = pd.read_csv("customer_info_1000.csv")

df = tickets.merge(customers[["customer_id", "country", "city"]], on="customer_id", how="left")

country_to_region = {
    "United States": "NA",  "Canada": "NA",    "Mexico": "NA",
    "Germany": "EU",        "France": "EU",    "United Kingdom": "EU",
    "Spain": "EU",          "Italy": "EU",     "Netherlands": "EU",
    "Sweden": "EU",         "Poland": "EU",    "Belgium": "EU",
    "China": "APAC",        "Japan": "APAC",   "India": "APAC",
    "Australia": "APAC",    "South Korea": "APAC", "Singapore": "APAC",
    "Brazil": "LATAM",      "Argentina": "LATAM",  "Colombia": "LATAM",
    "Chile": "LATAM",       "Peru": "LATAM",
    "South Africa": "MEA",  "Nigeria": "MEA",  "Kenya": "MEA",
    "Saudi Arabia": "MEA",  "UAE": "MEA",      "Egypt": "MEA",
}
mask = df["region"].isna()
df.loc[mask, "region"] = df.loc[mask, "country"].map(country_to_region)
df = df.dropna(subset=["region"])
df["csat_score"] = df["csat_score"].replace(0, np.nan)
df["year_month"]  = df["created_at"].dt.to_period("M").astype(str)
df["is_resolved"] = df["status"].isin(["resolved", "closed_no_action"])

resolved = df.dropna(subset=["resolution_time_hours"])
csat_df  = df.dropna(subset=["csat_score"])

print("Rows:", len(df))

Rows: 82706


## Mask sensitive columns

In [2]:
import hashlib

def hash_value(val):
    if pd.isna(val):
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()[:16]

pii_columns = ["customer_name", "contact_person", "email_address", "Address_line_1"]
for col in pii_columns:
    if col in customers.columns:
        customers[col] = customers[col].apply(hash_value)

print("PII columns masked.")


PII columns masked.


## Resolution time by priority

In [3]:
priority_resolution = (
    resolved
    .groupby("priority")["resolution_time_hours"]
    .median()
    .reindex(["urgent", "high", "medium", "low"])
)
priority_resolution

priority
urgent     4.875
high      14.730
medium    29.820
low       45.540
Name: resolution_time_hours, dtype: float64

## Resolution time by SLA plan

In [4]:
sla_resolution = (
    resolved
    .groupby("sla_plan")["resolution_time_hours"]
    .median()
    .sort_values()
)
sla_resolution

sla_plan
platinum    29.15
gold        29.73
standard    30.01
Name: resolution_time_hours, dtype: float64

## Monthly ticket volume

In [5]:
monthly = (
    df.groupby("year_month")
    .agg(n_tickets=("ticket_id", "count"), avg_csat=("csat_score", "mean"))
    .reset_index()
)
monthly.head(6)

,year_month,n_tickets,avg_csat
0,2022-01,1738,3.180500
1,2022-02,1536,3.122355
2,2022-03,1747,3.183740
3,2022-04,1723,3.224700
4,2022-05,1663,3.279931
5,2022-06,1688,3.176572


## Status breakdown

In [6]:
status_breakdown = df["status"].value_counts()
status_breakdown

status
resolved            41442
in_progress         16315
on_hold              8413
closed_no_action     8281
open                 8255
Name: count, dtype: int64

## Average CSAT by SLA plan

In [7]:
csat_by_sla = csat_df.groupby("sla_plan")["csat_score"].mean().sort_values()
csat_by_sla

sla_plan
gold        3.190357
standard    3.202053
platinum    3.209040
Name: csat_score, dtype: float64

## Resolution time by region and SLA

In [8]:
region_sla = (
    resolved
    .groupby(["region", "sla_plan"])["resolution_time_hours"]
    .median()
    .unstack()
)
region_sla

sla_plan,gold,platinum,standard
region,,,
APAC,30.650,30.690,30.245
EU,28.600,28.120,29.880
LATAM,29.710,27.535,29.620
MEA,30.385,31.095,30.525
NA,29.470,28.740,27.190


## Ticket volume by channel and issue type

In [9]:
channel_counts = df["channel"].value_counts()
issue_counts   = df["issue_type"].value_counts()
print("By channel:\n", channel_counts)
print()
print("By issue type:\n", issue_counts)

By channel:
 channel
email               16638
phone_transcript    16635
in_app              16548
web_form            16524
chat                16361
Name: count, dtype: int64

By issue type:
 issue_type
how_to              10513
account_access      10411
performance         10405
other               10312
billing_problem     10305
security_concern    10303
feature_request     10302
bug                 10155
Name: count, dtype: int64


## CSAT and reopen rate by issue type

In [10]:
issue_stats = (
    df.groupby("issue_type")
    .agg(
        avg_csat        = ("csat_score", "mean"),
        reopen_rate     = ("reopened", "mean"),
        median_res_time = ("resolution_time_hours", "median"),
    )
    .sort_values("avg_csat")
)
issue_stats

,avg_csat,reopen_rate,median_res_time
issue_type,,,
security_concern,2.799200,0.052509,29.810
account_access,2.810515,0.053213,29.505
billing_problem,2.821424,0.048229,30.370
performance,2.828194,0.053051,29.695
bug,3.420095,0.051896,29.180
other,3.438238,0.051978,29.360
feature_request,3.725764,0.050184,30.515
how_to,3.754712,0.046989,30.535


## CSAT by customer segment and region

In [11]:
csat_heatmap = (
    csat_df
    .groupby(["customer_segment", "region"])["csat_score"]
    .mean()
    .unstack()
)
csat_heatmap

region,APAC,EU,LATAM,MEA,NA
customer_segment,,,,,
education,3.206514,3.200967,3.260406,3.183397,3.330275
enterprise,3.198705,3.201054,3.168036,3.232105,3.310000
individual,3.194081,3.229145,3.225299,3.212153,3.241758
non_profit,3.188507,3.174124,3.204513,3.168999,3.168317
small_business,3.182732,3.191762,3.186119,3.167244,3.295455
